# Manual Evaluation (Cross-Model Rubric)

Per `docs/methodology.md`, BART and the prompted LLM aren't comparable via
a single automatic metric -- BART summarizes single reviews against a real
reference, the LLM synthesizes whole product batches with no reference at
that granularity. The manual rubric (`evaluation/manual_evaluation.py`) is
the fair head-to-head: a human scores each model's output on its own terms.

**What this notebook does:**
1. Samples 10 products and generates both models' summaries.
2. Saves a CSV with empty scoring columns for you to fill in by hand
   (Google Sheets or any spreadsheet app is easiest).
3. Loads your filled-in scores back and builds the final `ManualScore`
   records via the existing `evaluation/manual_evaluation.py` scaffold.

In [ ]:
# 1. Environment setup: mount Drive, clone/pull repo, install deps, load keys,
# restore product batches + BART checkpoint from Drive backup if not present locally.
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_DIR = '/content/Review-Summarization-LLM'
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    %cd /content
    !git clone https://github.com/jrsheffie/Review-Summarization-LLM.git
    %cd {REPO_DIR}

!pip install -q -r requirements.txt
!pip install -q -U torchao

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
github_token = userdata.get('GITHUB_TOKEN')
!git config --global user.email "jrsheffie@gmail.com"
!git config --global user.name "jrsheffie"

BACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'

if not os.path.exists('data/processed/product_batches.json'):
    print('Restoring processed data from Drive backup...')
    !mkdir -p data/processed data/raw
    !cp {BACKUP_DIR}/clean_reviews.csv {BACKUP_DIR}/train.csv {BACKUP_DIR}/val.csv {BACKUP_DIR}/test.csv {BACKUP_DIR}/product_batches.json data/processed/ 2>/dev/null
    !cp {BACKUP_DIR}/Reviews.csv data/raw/ 2>/dev/null

if not os.path.exists('outputs/bart_lora_30k/final'):
    print('Restoring BART checkpoint from Drive backup...')
    !mkdir -p outputs/bart_lora_30k/final
    !cp -r {BACKUP_DIR}/bart_lora_30k/* outputs/bart_lora_30k/final/

assert os.path.exists('data/processed/product_batches.json'), "product_batches.json still missing."
assert os.path.exists('outputs/bart_lora_30k/final'), "BART checkpoint still missing."
print('Environment ready.')

In [ ]:
# 2. Sample 10 products and generate both models' summaries.
import json
import random
import sys

import torch
from transformers import BartTokenizer, BartForConditionalGeneration
from peft import PeftModel

sys.path.insert(0, '.')
from models.llm_prompting import summarize_product
from models.config import LLMConfig

SEED = 42
N_PRODUCTS = 10

with open('data/processed/product_batches.json') as f:
    all_batches = json.load(f)

random.seed(SEED)
sample_batches = random.sample(all_batches, N_PRODUCTS)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')
bart_tokenizer = BartTokenizer.from_pretrained('outputs/bart_lora_30k/final')
bart_model = PeftModel.from_pretrained(base_model, 'outputs/bart_lora_30k/final')
bart_model.to(device)
bart_model.eval()

llm_config = LLMConfig()

rows = []
for i, batch in enumerate(sample_batches):
    product_id = batch.get('product_id')

    # BART: summarize the single most-helpful review in the batch (BART's task granularity)
    representative_review = max(batch['reviews'], key=lambda r: r.get('helpful_votes', 0))
    rep_text = representative_review.get('text', '')

    inputs = bart_tokenizer(rep_text, return_tensors='pt', max_length=1024, truncation=True).to(device)
    with torch.no_grad():
        output_ids = bart_model.generate(**inputs, max_length=64, num_beams=4, early_stopping=True)
    bart_summary = bart_tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # LLM: summarize the full multi-review batch (the LLM's task granularity)
    llm_summary = summarize_product(batch, config=llm_config)

    all_reviews_text = "\n\n".join(
        f"({r.get('rating', '?')} stars) {r.get('text', '')}" for r in batch['reviews']
    )

    rows.append({
        'product_id': product_id,
        'all_reviews': all_reviews_text,
        'bart_input_review': rep_text,
        'bart_summary': bart_summary,
        'llm_summary': llm_summary,
        'bart_accuracy': '', 'bart_completeness': '', 'bart_readability': '',
        'bart_hallucination_free': '', 'bart_overall_quality': '', 'bart_notes': '',
        'llm_accuracy': '', 'llm_completeness': '', 'llm_readability': '',
        'llm_hallucination_free': '', 'llm_overall_quality': '', 'llm_notes': '',
    })
    print(f'{i + 1}/{N_PRODUCTS} done: {product_id}')

import pandas as pd
candidates_df = pd.DataFrame(rows)
os.makedirs('outputs/metrics', exist_ok=True)
candidates_df.to_csv('outputs/metrics/manual_eval_candidates.csv', index=False)
print('Saved outputs/metrics/manual_eval_candidates.csv -- fill in the score columns by hand.')

## Fill in your scores

Download `outputs/metrics/manual_eval_candidates.csv` (or open it in Google
Sheets) and, for each of the 10 rows, read `all_reviews` alongside
`bart_summary` and `llm_summary`, then fill in the `bart_*` and `llm_*`
columns (1-5 for each score field, a short note for each `*_notes` field --
see `evaluation/manual_evaluation.py`'s `ManualScore` fields for
definitions: accuracy, completeness, readability, hallucination_free,
overall_quality).

Once filled in, re-upload the CSV to the same path
(`outputs/metrics/manual_eval_candidates.csv`) before running the next
cell -- or just edit the file in place if you're working directly in the
Colab file browser.

In [ ]:
# 3. Load your filled-in scores and build the final ManualScore records for each model.
import pandas as pd

from evaluation.manual_evaluation import ManualScore, save_scores

filled_df = pd.read_csv('outputs/metrics/manual_eval_candidates.csv')

bart_scores = []
llm_scores = []

for _, row in filled_df.iterrows():
    bart_scores.append(ManualScore(
        product_id=row['product_id'],
        accuracy=int(row['bart_accuracy']),
        completeness=int(row['bart_completeness']),
        readability=int(row['bart_readability']),
        hallucination_free=int(row['bart_hallucination_free']),
        overall_quality=int(row['bart_overall_quality']),
        notes=str(row.get('bart_notes', '')),
    ))
    llm_scores.append(ManualScore(
        product_id=row['product_id'],
        accuracy=int(row['llm_accuracy']),
        completeness=int(row['llm_completeness']),
        readability=int(row['llm_readability']),
        hallucination_free=int(row['llm_hallucination_free']),
        overall_quality=int(row['llm_overall_quality']),
        notes=str(row.get('llm_notes', '')),
    ))

save_scores(bart_scores, 'outputs/metrics/manual_scores_bart.csv')
save_scores(llm_scores, 'outputs/metrics/manual_scores_llm.csv')

# Quick averaged comparison
import statistics

def avg(scores, field):
    return statistics.mean(getattr(s, field) for s in scores)

fields = ['accuracy', 'completeness', 'readability', 'hallucination_free', 'overall_quality']
print(f"{'Metric':<20}{'BART':>8}{'LLM':>8}")
for f in fields:
    print(f"{f:<20}{avg(bart_scores, f):>8.2f}{avg(llm_scores, f):>8.2f}")

## Next steps

Fold this averaged comparison table into `docs/evaluation_report.md`'s
cross-model comparison section, alongside the earlier LLM-judge findings
and BART's ROUGE/BERTScore numbers, to complete the analysis called for in
`docs/model_comparison.md`.